<a href="https://colab.research.google.com/github/arinjay-singh/econ3916-statistical-machine-learning/blob/main/Class%2012%20/%20class12_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Environment Initialization and Data Ingestion

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
path = 'Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(path)
df.head()

,Home_Value,Square_Footage,Property_Age,Distance_to_Transit,School_District_Rating
0,329705.74,1941.0,5.5,6.45,Excellent
1,183343.63,1364.3,35.2,2.15,Average
2,354551.73,2386.9,52.4,0.75,Good
3,325773.17,2192.1,50.2,5.25,Excellent
4,359743.12,3069.8,66.5,12.69,Excellent


Step 2: Defining the OLS Architecture via Patsy Formulas

In [5]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula = 'Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'

Step 3: Model Fitting and Diagnostic Extraction

In [6]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula, data=df)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Mon, 16 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        19:54:30   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

Step 4: The Machine Learning Pivot: Generating Predictions


In [9]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict()

Step 5: Calculating and Interpreting the Root Mean Squared Error

In [10]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df['Home_Value'], y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


In [11]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import plotly.express as px
import plotly.graph_objects as go

# ── LOAD & FIT (matching your existing pipeline exactly) ─────────────────────
path = 'Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(path)

formula = 'Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'
model   = smf.ols(formula=formula, data=df)
results = model.fit()

# ── EXTRACT RESIDUALS FROM STATSMODELS RESULTS OBJECT ────────────────────────
# smf.ols wraps sm.OLS; the RegressionResultsWrapper exposes the same attributes.
# results.fittedvalues → ŷ computed internally during .fit() — identical to
#   results.predict() but already cached; either works here.
# results.resid        → raw residuals e_i = y_i − ŷ_i, aligned to df's index.
fitted    = results.fittedvalues          # ŷ  (same as your y_pred)
residuals = results.resid                 # e_i = Home_Value_i − ŷ_i

# Confirm RMSE matches your Step 5 output
model_rmse = rmse(df['Home_Value'], fitted)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")

# ── OUTLIER DETECTION — 2σ THRESHOLD ─────────────────────────────────────────
# Standardize: z_i = e_i / σ(e).  |z| > 2 flags the extreme ~5% tail.
resid_std  = residuals.std()
z_scores   = residuals / resid_std
is_outlier = z_scores.abs() > 2          # boolean mask → drives color mapping

plot_df = pd.DataFrame({
    "fitted":     fitted,
    "residual":   residuals,
    "z_score":    z_scores,
    # Categorical label → Plotly maps it to color_discrete_map below
    "point_type": np.where(is_outlier, "Outlier (|z| > 2)", "Normal"),
})

# ── RESIDUAL FORENSICS SCATTER ────────────────────────────────────────────────
# color_discrete_map pins exact colors to each category label — no ambiguity.
fig = px.scatter(
    plot_df,
    x="fitted",
    y="residual",
    color="point_type",
    color_discrete_map={
        "Normal":            "#4a90d9",   # steel-blue
        "Outlier (|z| > 2)": "#DC143C",   # stark crimson
    },
    hover_data={
        "z_score":    ":.2f",
        "fitted":     ":,.0f",
        "residual":   ":,.0f",
        "point_type": False,
    },
    labels={
        "fitted":    "Fitted Values  ŷ  ($)",
        "residual":  "Residuals  e = y − ŷ  ($)",
        "point_type": "",
    },
    title="Residual Forensics Dashboard — Zillow Hedonic Pricing OLS",
    opacity=0.72,
)

# ── ZERO LINE: E[e|X] = 0 ────────────────────────────────────────────────────
# Horizontal dashed line at y=0 — the OLS theoretical expectation.
# Systematic drift above/below signals omitted variables or misspecification.
fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="#888888",
    line_width=1.5,
    annotation_text="E[e|X] = 0",
    annotation_position="top right",
    annotation_font_color="#888888",
)

# ── ±2σ REFERENCE BANDS ───────────────────────────────────────────────────────
# Dotted crimson lines mark the outlier threshold — anything beyond is flagged.
for sign, label, pos in [(+1, "+2σ", "top right"), (-1, "−2σ", "bottom right")]:
    fig.add_hline(
        y=sign * 2 * resid_std,
        line_dash="dot",
        line_color="#DC143C",
        line_width=1,
        opacity=0.45,
        annotation_text=label,
        annotation_position=pos,
        annotation_font_color="#DC143C",
    )

# ── LAYOUT ────────────────────────────────────────────────────────────────────
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#0f1117",
    paper_bgcolor="#0f1117",
    font=dict(family="'IBM Plex Mono', monospace", size=12, color="#e0e0e0"),
    title_font_size=15,
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1
    ),
    xaxis=dict(
        showgrid=True, gridcolor="#1e1e2e", zeroline=False,
        tickformat="$,.0f",
    ),
    yaxis=dict(
        showgrid=True, gridcolor="#1e1e2e", zeroline=False,
        tickformat="$,.0f",
    ),
    margin=dict(l=70, r=40, t=90, b=60),
)

fig.update_traces(marker=dict(size=5, line=dict(width=0)))

# Diagnostic summary annotation — bottom-left corner
fig.add_annotation(
    text=(
        f"n={len(df):,}  |  Outliers (|z|>2): {is_outlier.sum()}  "
        f"|  RMSE: ${model_rmse:,.2f}  |  R²: {results.rsquared:.3f}"
    ),
    xref="paper", yref="paper",
    x=0.01, y=0.01,
    showarrow=False,
    font=dict(size=10, color="#aaaaaa"),
    align="left",
)

fig.show()


The Predictive RMSE is: $42,316.69
